# Семинар 4. Что модель выучила и где она ошибается?

Мы уже умеем обучать линейную регрессию и Ridge. Теперь исследуем готовую модель: насколько полезны её прогнозы, какие ошибки она допускает и как использует признаки.

**План:** R² → анализ ошибок и похожих домов → коэффициенты → их устойчивость → важность признаков.

Работаем с теми же домами Ames. Небольшие синтетические примеры помогут прочитать графики. Длинные ячейки построения графиков и бутстрапа подготовлены; в конце есть самостоятельная работа. Дополнительный блок о статистической значимости можно изучить отдельно.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    mean_absolute_percentage_error, r2_score,
)
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

sns.set_theme(style="darkgrid")

## 1. R²: насколько модель лучше прогноза одним средним?

MAE измеряет среднюю абсолютную ошибку, RMSE сильнее реагирует на крупные ошибки. Обе метрики выражены в единицах целевой переменной. Добавим сравнение с разбросом самих значений.

> **Коэффициент детерминации (coefficient of determination), R²** — метрика, сравнивающая сумму квадратов ошибок модели с суммой квадратов отклонений целевой переменной от её среднего на оцениваемой выборке.

$$R^2=1-\frac{\sum_{i=1}^{n}(y_i-\widehat y_i)^2}{\sum_{i=1}^{n}(y_i-\overline y)^2},\qquad \overline y=\frac{1}{n}\sum_{i=1}^{n}y_i.$$

**Где:** $n$ — число оцениваемых объектов; $i$ — номер объекта; $y_i$ — фактическое значение; $\widehat y_i$ — прогноз; $\overline y$ — среднее фактических значений **этой же выборки**. Числитель — **SSE (sum of squared errors)**, сумма квадратов ошибок модели; знаменатель — **SST (total sum of squares)**, сумма квадратов отклонений цели от среднего, то есть ошибка постоянного прогноза $\overline y$.

* **R² = 1:** все прогнозы точны.
* **R² = 0:** сумма квадратов ошибок такая же, как у прогноза средним.
* **R² < 0:** сумма квадратов ошибок больше, чем у прогноза средним.

Например, R² = 0,8 означает, что сумма квадратов ошибок составляет 20% от знаменателя формулы. Часто говорят: «модель объясняет 80% изменчивости цели». Это сравнение ошибок, **не 80% правильно предсказанных объектов и не доказательство причинной связи**.

> **R² показывает, насколько модель сократила квадратичную ошибку относительно прогноза средним. Необъяснённая часть может содержать и случайный шум, и закономерности, которые модель пока не нашла.**

In [ ]:
# Synthetic prices, in thousands of dollars.
y_small = np.array([10., 20., 30., 40.])
toy_predictions = {
    "Прогноз средним": np.full(4, y_small.mean()),
    "Близкий прогноз": np.array([15., 20., 30., 35.]),
    "Плохой прогноз": np.array([40., 30., 20., 10.]),
}

sst = np.sum((y_small - y_small.mean()) ** 2)
toy_rows = []
fig, axs = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
for ax, (name, prediction) in zip(axs, toy_predictions.items()):
    sse = np.sum((y_small - prediction) ** 2)
    r2_manual = 1 - sse / sst
    toy_rows.append({"Прогноз": name, "SSE": sse, "SST": sst,
                     "R² вручную": r2_manual, "R² sklearn": r2_score(y_small, prediction)})
    ax.scatter(y_small, prediction, color="black", s=50, label="Объекты")
    ax.plot([5, 45], [5, 45], "--", color="tab:blue", label="Точное совпадение")
    ax.set(xlim=(5, 45), ylim=(5, 45), title=f"{name}: R² = {r2_manual:.1f}",
           xlabel="Фактическая цена, тыс. $", ylabel="Прогноз, тыс. $")
    ax.set_aspect("equal", adjustable="box")
axs[0].legend(fontsize=9)
plt.show()
display(pd.DataFrame(toy_rows).set_index("Прогноз"))

**Среднее в R² и наш baseline — разные вещи.** На validation формула использует `y_val.mean()`. Наш реальный baseline берёт среднее **train**, поскольку цены новых домов заранее неизвестны. Поэтому его R² на validation может быть отрицательным. Среднее validation нужно только для расчёта метрики, его не передают модели.

R² удобно сравнивать на одной проверочной выборке; оно не показывает размер ошибки в долларах. Продолжаем смотреть и MAE/RMSE. R² не обязано совпадать с квадратом корреляции между целью и прогнозом.

Если все $y_i$ одинаковы, знаменатель равен нулю: исходная формула не определена. `r2_score` по умолчанию заменяет такие результаты на 1 для точного прогноза и 0 для неточного. Это правило библиотеки. [Документация R²](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html).

<details>
<summary><strong>Для любознательных: объяснённая изменчивость, дисперсия и случайный шум</strong></summary>

**Почему говорят «модель объясняет изменчивость»?**

В формуле R² сравниваются суммы квадратов отклонений. Если ошибка прогноза средним равна 1000, а ошибка модели — 200, то R² = 0,8: модель сократила квадратичную ошибку на 80%. «Объяснила» здесь означает статистическое описание различий между объектами, а не установление их причин.

Сумма квадратов отклонений сама по себе не равна дисперсии. Если разделить её на число объектов, получим дисперсию с делителем $n$:

$$s_n^2=\frac{1}{n}\sum_{i=1}^{n}(y_i-\overline y)^2,\qquad R^2=1-\frac{\mathrm{MSE}}{s_n^2}.$$

**Где:** $n$ — число оцениваемых объектов; $y_i$ — фактическая цель объекта $i$; $\overline y$ — её среднее на этой выборке; $s_n^2$ — дисперсия с делителем $n$; $\mathrm{MSE}=\frac{1}{n}\sum_i(y_i-\widehat y_i)^2$ — средняя квадратичная ошибка; $\widehat y_i$ — прогноз. Здесь делитель $n$, а не $n-1$, чтобы равенство точно соответствовало формуле R².

**Особый случай: МНК со свободным членом на обучающих данных.**

Для него выполняется точное разложение:

$$\underbrace{\sum_i(y_i-\overline y)^2}_{\text{общая изменчивость}}=\underbrace{\sum_i(\widehat y_i-\overline y)^2}_{\text{изменчивость прогнозов}}+\underbrace{\sum_i(y_i-\widehat y_i)^2}_{\text{оставшаяся ошибка}}.$$

**Где:** $y_i$ — обучающие значения цели; $\widehat y_i$ — прогнозы обученного МНК для тех же объектов; $\overline y$ — средняя обучающая цель; $i$ — номер объекта. Все три выражения — суммы квадратов; для перехода к средним квадратам делим все части на $n$.

Именно здесь «доля объяснённой изменчивости» читается буквально из разложения. Для Ridge и для проверки на новых данных это равенство в общем случае не выполняется. Там R² по-прежнему показывает относительное уменьшение квадратичной ошибки. [Penn State: разложение сумм квадратов](https://online.stat.psu.edu/stat501/Lesson02).

**Как это связано с шумом цели?**

Представим цель как среднюю закономерность и отклонение от неё:

$$Y=f(X)+\varepsilon,\qquad f(X)=\mathbb E[Y\mid X],\qquad \mathbb E[\varepsilon\mid X]=0.$$

**Где:** $X$ — доступные признаки; $Y$ — случайная целевая переменная; $f(X)$ — средняя цель при данных признаках; $\mathbb E$ — математическое ожидание; $\varepsilon$ — случайное отклонение от этой средней зависимости. Последнее равенство означает, что при фиксированных признаках у отклонения нет систематического сдвига.

Даже идеально найденная функция $f$ не предскажет каждое такое отклонение. При конечной ненулевой дисперсии цели для генеральной совокупности получаем:

$$\operatorname{Var}(Y)=\operatorname{Var}(f(X))+\operatorname{Var}(\varepsilon),\qquad R^2_{\max}=1-\frac{\operatorname{Var}(\varepsilon)}{\operatorname{Var}(Y)}=\frac{\operatorname{Var}(f(X))}{\operatorname{Var}(Y)}.$$

**Где:** $\operatorname{Var}$ — дисперсия; $Y,f(X),\varepsilon$ — величины из предыдущей формулы; $R^2_{\max}$ — наилучшее качество в генеральной совокупности по данным признакам и при том же распределении объектов. Это предел для оптимального прогноза по доступным признакам; линейная модель может его не достичь, если $f$ нелинейна. Результат на отдельной небольшой выборке может отличаться от качества в генеральной совокупности. Нормальность и одинаковая дисперсия ошибок для этого разложения не требуются.

Рассмотрим числовой пример **для генеральной совокупности**, а не для одной проверочной выборки. Допустим, нам отдельно известно, что общая дисперсия цели равна 100, а дисперсия шума — 20. Идеальный прогноз тогда имеет R² = 0,8. У реальной модели с R² = 0,6 оставшаяся средняя квадратичная ошибка равна 40: в этом примере 20 приходится на шум и ещё 20 — на несовершенство модели. Но **из одного измеренного R² нельзя узнать такое разделение**.

Шум определяется относительно доступных признаков. Если известна только площадь дома, неизвестное состояние ремонта входит в непредсказуемую часть. Информация о ремонте может сделать часть прежнего шума предсказуемой. [CMU: оптимальный прогноз и непредсказуемая ошибка](https://www.stat.cmu.edu/~siva/teaching/462/lec2.pdf).

**На какие вопросы отвечает R²?**

* Насколько модель улучшает прогноз относительно одной общей средней величины?
* Какая из моделей лучше предсказывает цель на **одних и тех же отложенных объектах**, если важна квадратичная ошибка?
* Помогло ли изменение признаков или модели на validation? Рост R² на train при добавлении признаков в МНК сам по себе не доказывает улучшения обобщения.

R² не определяет важность отдельного признака и не доказывает причинность. Оно также не отвечает, приемлема ли ошибка в долларах: для этого нужны MAE/RMSE и требования задачи. При одинаковой MSE выборка с большей дисперсией цен даст более высокое R². Поэтому широкий рынок и небольшой однородный сегмент нельзя сравнивать только по этому числу. [sklearn: R²](https://scikit-learn.org/stable/modules/model_evaluation.html#r2-score).

</details>

## 2. Возвращаемся к знакомой модели

| Столбец | Что означает | Единицы после подготовки |
|---|---|---|
| `GrLivArea` | Надземная жилая площадь | м² |
| `GarageArea` | Площадь гаража | м² |
| `TotalBsmtSF` | Площадь подвала | м² |
| `YearBuilt` | Год постройки | год |
| `LotFrontage` | Длина границы участка вдоль улицы | м |
| `Neighborhood` | Район | категория |
| `SalePrice` | Целевая переменная: цена продажи | $ |

Сохраняем загрузку, перевод единиц, признаки и разбиение предыдущего семинара. Подготовку обучаем только на train. **Диагностику проводим на validation; test остаётся закрытым.**

In [ ]:
DATA_URL = (
    "https://www.dropbox.com/scl/fi/65u0zpc8dweb9rx1tlzol/train_sem2.csv"
    "?rlkey=5vi4mt47fs6magpldgj5wp2u9&dl=1"
)
raw_data = pd.read_csv(DATA_URL)
print(f"Загружено {len(raw_data)} домов.")

In [ ]:
BASE_FEATURES = ["GrLivArea", "GarageArea", "TotalBsmtSF"]
numeric_features = BASE_FEATURES + ["YearBuilt", "LotFrontage"]
categorical_features = ["Neighborhood"]
features = numeric_features + categorical_features

data = raw_data.copy()
data[BASE_FEATURES + ["LotArea"]] *= 0.09290304
data["LotFrontage"] *= 0.3048
assert (data["SalePrice"] > 0).all()

In [ ]:
train_val, test = train_test_split(data, test_size=0.2, random_state=42)
train, validation = train_test_split(train_val, test_size=0.25, random_state=42)

train = train.copy()
validation = validation.copy()
test = test.copy()
y_train = train["SalePrice"]
y_val = validation["SalePrice"]

print(f"Train: {len(train)}; validation: {len(validation)}; test: {len(test)}.")
print("Test отложен: сегодня его не исследуем и не оцениваем.")

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
     categorical_features),
])

In [ ]:
def make_model(estimator):
    return Pipeline([
        ("preprocess", clone(preprocessor)),
        ("regression", estimator),
    ])

Повторим МНК и `Ridge(alpha=10)`. Здесь `alpha` фиксирован для демонстрации; это не утверждение, что 10 — лучшая настройка. `clone` создаёт необученную копию подготовки для каждой модели.

Основной объект исследования ниже — Ridge. МНК оставим для сравнения. Неизвестный район validation, как и раньше, кодируется нулями (`handle_unknown="ignore"`); предупреждения в этом ноутбуке отключены, а способ кодирования остаётся прежним.

К знакомым метрикам добавляем R². MAPE выводится в процентах; в этих данных цены положительны. При нулевых или близких к нулю целях обычная MAPE не подходит.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE, $": mean_absolute_error(y_true, y_pred),
        "RMSE, $": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE, %": 100 * mean_absolute_percentage_error(y_true, y_pred),
        "R²": r2_score(y_true, y_pred),
    }

ols_model = make_model(LinearRegression()).fit(train[features], y_train)
ridge_model = make_model(Ridge(alpha=10)).fit(train[features], y_train)
model = ridge_model
pred_val = model.predict(validation[features])
pred_train = model.predict(train[features])

quality = pd.DataFrame({
    "Среднее train → validation": regression_metrics(y_val, np.full(len(y_val), y_train.mean())),
    "МНК → validation": regression_metrics(y_val, ols_model.predict(validation[features])),
    "Ridge → validation": regression_metrics(y_val, pred_val),
    "Ridge → train": regression_metrics(y_train, pred_train),
}).T
display(quality.round({"MAE, $": 0, "RMSE, $": 0, "MAPE, %": 2, "R²": 6}))

Сравните строки validation: дала ли модель улучшение относительно baseline? Одинаков ли порядок МНК и Ridge по разным метрикам?

Ошибка train описывает подгонку к знакомым объектам. Для оценки работы на новых объектах используем validation. R² зависит также от разброса цели: его различие между train и validation нельзя читать как чистую меру переобучения.

## 3. Где модель ошибается?

> **Остаток (residual)** — разность фактического значения целевой переменной и прогноза модели.

$$e_i=y_i-\widehat y_i.$$

**Где:** $e_i$ — остаток для объекта $i$; $y_i$ — фактическая цена; $\widehat y_i$ — прогноз. Все три величины здесь в долларах. **Положительный остаток — заниженный прогноз; отрицательный — завышенный.**

Сначала — три синтетических примера. В верхнем ряду данные и обученная прямая, в нижнем — остатки против признака. Модель везде линейная, но закономерности в данных разные.

In [ ]:
#@title Три рисунка остатков на синтетике { display-mode: "form" }
rng_patterns = np.random.default_rng(12)
x_pattern = np.linspace(-3, 3, 240)
eps_pattern = rng_patterns.normal(size=len(x_pattern))
pattern_targets = {
    "Линейная зависимость": 3 + 2 * x_pattern + 0.7 * eps_pattern,
    "Пропущенная кривизна": 3 + 2 * x_pattern + x_pattern ** 2 + 0.7 * eps_pattern,
    "Растущий разброс": 3 + 2 * x_pattern + (0.2 + 0.45 * (x_pattern + 3)) * eps_pattern,
}
fig, axs = plt.subplots(2, 3, figsize=(14, 7), layout="constrained")
for col, (name, target) in enumerate(pattern_targets.items()):
    fitted = LinearRegression().fit(x_pattern[:, None], target)
    predicted = fitted.predict(x_pattern[:, None])
    axs[0, col].scatter(x_pattern, target, color="black", alpha=0.2, label="Наблюдения")
    axs[0, col].plot(x_pattern, predicted, color="tab:orange", label="Линейный прогноз")
    axs[0, col].set(title=name, xlabel="Признак x, усл. ед.", ylabel="Цель y, усл. ед.")
    axs[1, col].scatter(x_pattern, target - predicted, color="black", alpha=0.2)
    axs[1, col].axhline(0, color="tab:orange", linestyle="--")
    axs[1, col].set(xlabel="Признак x, усл. ед.", ylabel="Остаток y − ŷ, усл. ед.")
axs[0, 0].legend(fontsize=9)
plt.show()

Дуга в остатках подсказывает, что прямая не описала форму зависимости. «Веер» показывает изменение разброса ошибок — **гетероскедастичность (heteroscedasticity)**. Здесь причины известны, поскольку мы сами создали данные.

На реальных данных рисунок — повод проверить объяснение, а не готовый диагноз. Отсутствие заметного рисунка тоже не доказывает, что вся закономерность найдена.

> **Остатки — не обязательно чистый шум.** В них могут оставаться пропущенные зависимости, последствия неполных данных и ошибки измерений.

[NIST: графический анализ остатков](https://www.itl.nist.gov/div898/handbook/pmd/section4/pmd44.htm).

In [ ]:
errors = validation[["Id"] + features + ["SalePrice"]].copy()
errors["Прогноз, $"] = pred_val
errors["Остаток, $"] = y_val.to_numpy() - pred_val
errors["Абс. ошибка, $"] = np.abs(errors["Остаток, $"])

fig, axs = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
axs[0].scatter(y_val, pred_val, color="black", alpha=0.2)
limits = [min(y_val.min(), pred_val.min()), max(y_val.max(), pred_val.max())]
axs[0].plot(limits, limits, "--", color="tab:orange", label="Точное совпадение")
axs[0].set(xlabel="Фактическая цена, $", ylabel="Прогноз, $", title="Ridge: validation")
axs[0].legend(fontsize=9)
axs[1].scatter(pred_val, errors["Остаток, $"], color="black", alpha=0.2)
axs[1].set(xlabel="Прогноз, $", ylabel="Остаток y − ŷ, $", title="Ошибки против прогноза")
axs[2].scatter(errors["GrLivArea"], errors["Остаток, $"], color="black", alpha=0.2)
axs[2].set(xlabel="Жилая площадь, м²", ylabel="Остаток y − ŷ, $", title="Ошибки против площади")
for ax in axs[1:]:
    ax.axhline(0, color="tab:orange", linestyle="--")
for ax in axs:
    ax.ticklabel_format(style="plain", axis="both")
plt.show()

### Что видно на этих графиках

Диапазон прогнозов уже диапазона фактических цен: наиболее дорогие дома на левом графике оказываются заметно ниже линии точного совпадения. Модель **занижает их стоимость**. На графике остатков это большие положительные значения.

Разброс ошибок у больших домов и больших прогнозируемых цен заметнее. Это похоже на рассмотренный «веер»: одна MAE скрывает неодинаковую точность для разных объектов. При этом есть и крупные отрицательные остатки — модель не просто занижает все цены на одну и ту же сумму.

**Что проверить:** сначала посмотреть ошибки по группам площади, затем сравнить конкретные дома с похожими. Возможные эксперименты — добавить признаки качества дома, проверить нелинейную зависимость цены от площади, отдельно изучить сегмент дорогих домов. Это гипотезы для проверки на validation, а не установленная причина по одному рисунку.

### Средняя ошибка может скрывать разные группы

Разделим дома на три группы по площади. Границы возьмём из train: примерно треть обучающих домов попадёт в каждую группу. В validation размеры групп могут отличаться.

**MAE группы** показывает величину ошибок; **средний остаток** — направление: положительный означает занижение в среднем. Остатки разных знаков могут взаимно сокращаться, поэтому средний остаток около нуля ещё не означает маленькую MAE.

In [ ]:
area_cutoffs = train["GrLivArea"].quantile([1 / 3, 2 / 3]).to_numpy()
area_labels = [f"До {area_cutoffs[0]:.0f} м²",
               f"{area_cutoffs[0]:.0f}–{area_cutoffs[1]:.0f} м²",
               f"Свыше {area_cutoffs[1]:.0f} м²"]
errors["Группа площади"] = pd.cut(
    errors["GrLivArea"], bins=[-np.inf, *area_cutoffs, np.inf], labels=area_labels,
)
group_summary = errors.groupby("Группа площади", observed=True).agg(
    n=("Остаток, $", "size"),
    MAE=("Абс. ошибка, $", "mean"),
    mean_residual=("Остаток, $", "mean"),
).rename(columns={"n": "Число домов", "MAE": "MAE, $", "mean_residual": "Средний остаток, $"})
display(group_summary.round(1))

fig, axs = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
labels_with_n = [f"{label}\nn = {int(row['Число домов'])}" for label, row in group_summary.iterrows()]
for ax, metric, color in zip(axs, ["MAE, $", "Средний остаток, $"], ["tab:blue", "tab:orange"]):
    ax.bar(labels_with_n, group_summary[metric], color=color, alpha=0.8)
    ax.axhline(0, color="black", linewidth=1)
    ax.set(xlabel="Группа по жилой площади", ylabel=metric, title=f"Validation: {metric}")
plt.show()

У больших домов MAE около 35,9 тыс. долларов, у небольших — около 14,9 тыс. В средней группе средний остаток отрицательный (примерно −5,2 тыс.): прогнозы в среднем завышены. В большой группе он положительный (примерно +5,9 тыс.): прогнозы в среднем занижены.

По всей validation средний остаток всего около +278 долларов. Значит, ошибки разных направлений взаимно сокращаются, хотя по группам проблема заметна. Простое прибавление одной константы ко всем прогнозам такую картину не исправит.

Крупные дома дороже, поэтому рядом с ошибкой в долларах полезно сравнить и относительную ошибку по группам. Если важна точность именно для дорогих домов, следует отдельно сформулировать требование к этому сегменту.

### Разбираем пять самых неточных прогнозов через похожие дома

Цена сама по себе мало что говорит без контекста. Для каждого проблемного прогноза найдём **ближайшие по признакам дома из train** и сравним цены и характеристики. Это исследование ошибок готовой модели, а не ещё одна обучаемая модель.

Вспомним расстояние между векторами из линейной алгебры. Сначала оставляем дома того же района, затем считаем евклидово расстояние по пяти числовым признакам, подготовленным обученными на train импутером и `StandardScaler`:

$$d(a,b)=\sqrt{\sum_{j=1}^{5}(z_{aj}-z_{bj})^2}.$$

**Где:** $a$ — исследуемый дом validation; $b$ — кандидат из train того же района; $z_{aj},z_{bj}$ — стандартизованные значения числового признака $j$ после заполнения пропусков; $d$ — расстояние без единиц. Чем оно меньше, тем ближе описания по выбранным признакам. Цену и `Id` при поиске **не используем**.

Выберем до $n$ ближайших объектов; по умолчанию $n=10$. Если в train меньше домов этого района, покажем все доступные и их число. **Ближайшие не обязательно достаточно похожи:** проверяем расстояния и сами характеристики. Сильная корреляция признаков тоже влияет на выбранное расстояние, поэтому это простой способ получить контекст, а не универсальная оценка сходства.

In [ ]:
worst_indices = errors.nlargest(5, "Абс. ошибка, $").index.to_list()
display(errors.loc[worst_indices, [
    "Id", "Neighborhood", "GrLivArea", "SalePrice", "Прогноз, $", "Остаток, $"
]].rename(columns={"SalePrice": "Цена, $", "GrLivArea": "Жилая площадь, м²"}).round(1))

In [ ]:
# The fitted preprocessing is reused; no fit is performed here.
neighbor_numeric_transformer = model.named_steps["preprocess"].named_transformers_["num"]
neighbor_train_z = pd.DataFrame(
    neighbor_numeric_transformer.transform(train[numeric_features]),
    index=train.index, columns=numeric_features,
)

def find_similar_houses(case_index, n_neighbors=10):
    query = validation.loc[[case_index]]
    candidates = train.index[train["Neighborhood"] == query["Neighborhood"].iloc[0]]
    query_z = neighbor_numeric_transformer.transform(query[numeric_features])[0]
    distances = np.sqrt(((neighbor_train_z.loc[candidates] - query_z) ** 2).sum(axis=1))
    nearest = distances.sort_values(kind="stable").head(n_neighbors)
    neighbors = train.loc[nearest.index, ["Id"] + numeric_features + ["SalePrice"]].copy()
    neighbors["Расстояние"] = nearest
    return neighbors, len(candidates)

neighbor_overview_rows = []
for case_index in worst_indices:
    neighbors, available = find_similar_houses(case_index, n_neighbors=10)
    query = validation.loc[case_index]
    neighbor_overview_rows.append({
        "Id": int(query["Id"]), "Район": query["Neighborhood"],
        "Выбрано / доступно": f"{len(neighbors)} / {available}",
        "Цена объекта, $": query["SalePrice"],
        "Медиана цен соседей, $": neighbors["SalePrice"].median(),
        "Q1 цен соседей, $": neighbors["SalePrice"].quantile(.25),
        "Q3 цен соседей, $": neighbors["SalePrice"].quantile(.75),
    })
neighbor_overview = pd.DataFrame(neighbor_overview_rows).set_index("Id")
display(neighbor_overview.round(0))

In [ ]:
#@title Пять домов на фоне train { display-mode: "form" }
scatter_features = ["GrLivArea", "YearBuilt", "GarageArea"]
scatter_labels = ["Жилая площадь, м²", "Год постройки", "Площадь гаража, м²"]
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
for ax, feature, label in zip(axs, scatter_features, scatter_labels):
    ax.scatter(train[feature], train["SalePrice"], color="black", alpha=.15, label="Train")
    highlighted = validation.loc[worst_indices]
    ax.scatter(highlighted[feature], highlighted["SalePrice"], color="tab:red", s=75,
               marker="*", label="5 крупнейших ошибок validation", zorder=3)
    for order, (_, row) in enumerate(highlighted.iterrows()):
        ax.annotate(str(int(row["Id"])), (row[feature], row["SalePrice"]),
                    xytext=(6, 8 if order % 2 == 0 else -14), textcoords="offset points", fontsize=9)
    ax.set(xlabel=label, ylabel="Фактическая цена, $", title="Положение проблемных для модели объектов")
axs[0].legend(fontsize=8)
plt.show()

### Сравнение одного объекта с его соседями

Ниже разобран первый дом из списка. В таблице — его характеристики и статистики выбранных соседей. Статистики считаются по исходным значениям без заполнения пропусков; колонка «Непустых» показывает, сколько значений использовано. При расчёте расстояния пропуски заполняются медианами train, как при прогнозировании.

На каждом scatterplot серые точки — train того же района, синие — выбранные соседи. Красная звезда — фактическая цена исследуемого дома, оранжевый крест — прогноз для него. У звезды и креста одинаковое значение признака; вертикальный отрезок между ними показывает ошибку.

Каждый график показывает только одну координату. Соседи выбираются одновременно по пяти числовым признакам, поэтому близость на одном рисунке ещё не означает близость по всему описанию.

In [ ]:
#@title Функция сравнения с соседями { display-mode: "form" }
comparison_columns = numeric_features + ["SalePrice"]
comparison_labels = ["Жилая площадь, м²", "Гараж, м²", "Подвал, м²",
                     "Год постройки", "Граница участка, м", "Цена, $"]

def show_house_context(case_index, n_neighbors=10):
    query = validation.loc[case_index]
    neighbors, available = find_similar_houses(case_index, n_neighbors)
    forecast = errors.loc[case_index, "Прогноз, $"]
    print(f"Id {int(query['Id'])} · район {query['Neighborhood']} · запрошено {n_neighbors}, "
          f"выбрано {len(neighbors)}, доступно в train {available}")
    print(f"Цена: {query['SalePrice']:,.0f} $ · прогноз: {forecast:,.0f} $ · "
          f"остаток: {query['SalePrice'] - forecast:+,.0f} $")
    if neighbors.empty:
        print("В train нет домов этого района: подобрать соседей по заданному правилу нельзя.")
        return
    print(f"Расстояния выбранных соседей: {neighbors['Расстояние'].min():.2f}–"
          f"{neighbors['Расстояние'].max():.2f}. Меньше означает ближе по пяти признакам.")
    context = pd.DataFrame({
        "Объект": query[comparison_columns].astype(float),
        "Непустых": neighbors[comparison_columns].count(),
        "Медиана соседей": neighbors[comparison_columns].median(),
        "Q1": neighbors[comparison_columns].quantile(.25),
        "Q3": neighbors[comparison_columns].quantile(.75),
        "Минимум": neighbors[comparison_columns].min(),
        "Максимум": neighbors[comparison_columns].max(),
    })
    context.index = comparison_labels
    display(context.round(1))

    local_train = train[train["Neighborhood"] == query["Neighborhood"]]
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, feature, label in zip(axs, scatter_features, scatter_labels):
        ax.scatter(local_train[feature], local_train["SalePrice"], color="black", alpha=.18,
                   label="Train того же района")
        ax.scatter(neighbors[feature], neighbors["SalePrice"], color="tab:blue", s=55,
                   edgecolor="white", linewidth=.5, label="Выбранные соседи", zorder=3)
        ax.plot([query[feature], query[feature]], [query["SalePrice"], forecast],
                color="tab:orange", linestyle="--", linewidth=1.5)
        ax.scatter([query[feature]], [query["SalePrice"]], color="tab:red", marker="*", s=180,
                   label="Объект: фактическая цена", zorder=5)
        ax.scatter([query[feature]], [forecast], color="tab:orange", marker="x", s=110,
                   linewidth=2, label="Объект: прогноз", zorder=5)
        ax.set(xlabel=label, ylabel="Цена, $", title=f"Id {int(query['Id'])}: {query['Neighborhood']}")
    axs[0].legend(fontsize=8, loc="best")
    fig.tight_layout()
    plt.show()

In [ ]:
show_house_context(worst_indices[0], n_neighbors=10)

Выберите любой из пяти домов и число соседей, затем нажмите **«Показать сравнение»**. Проверьте, сохраняется ли наблюдение при $n=5$, $10$ и $20$: при увеличении $n$ в сравнение попадают всё менее близкие объекты.

Элементы управления работают при запущенном Python-ядре. Тот же разбор можно вызвать обычной строкой `show_house_context(worst_indices[4], n_neighbors=10)`; число в квадратных скобках выбирает объект из списка от 0 до 4.

In [ ]:
#@title Выберите дом и число соседей { display-mode: "form" }
import ipywidgets as widgets

case_options = [
    (f"Id {int(validation.loc[idx, 'Id'])} · {validation.loc[idx, 'Neighborhood']}", idx)
    for idx in worst_indices
]
case_selector = widgets.Dropdown(options=case_options, description="Объект:",
                                 layout=widgets.Layout(width="360px"))
neighbor_selector = widgets.IntSlider(value=10, min=5, max=20, step=1,
                                      description="Соседей:", continuous_update=False)
neighbor_explorer = widgets.interactive(
    show_house_context, {"manual": True, "manual_name": "Показать сравнение"},
    case_index=case_selector, n_neighbors=neighbor_selector,
)
display(neighbor_explorer)

### Что даёт сравнение при n = 10

| Дом | Наблюдение по данным | Что проверить дальше |
|---|---|---|
| **1170, NoRidge** | Дом заметно больше большинства выбранных соседей. Его цена 625 тыс., медиана соседей около 324 тыс.; при этом среди соседей есть цена 745 тыс. | Для больших домов изучить форму зависимости от площади и недостающие признаки качества. Нельзя заключить, что высокая цена сама по себе ошибочна. |
| **179, StoneBr** | Гараж и подвал больше, чем у всех десяти выбранных соседей. Прямых аналогов по этим характеристикам мало. | Проверить, достаточно ли линейного вклада каждой площади и хватает ли объектов такого типа в train. |
| **1182, Crawfor** | Исследуемый дом построен в 2008 году, многие соседи существенно старше. Их медиана не является готовой оценкой нового дома. | Проверить описание новизны и качества дома, зависимость эффекта года постройки от района. |
| **54, Veenker** | В train всего четыре дома этого района; у всех меньше гараж и подвал. | Отметить нехватку подходящих примеров. При возможности собрать данные, проверить результат на более широком, явно заданном наборе сравнения. |
| **1325, Somerst** | Цена 147 тыс. ниже цен всех десяти соседей (минимум 227 тыс., медиана около 250 тыс.), хотя описания сравнительно близки. Прогноз около 273 тыс. | Проверить запись, условия продажи и характеристики, отсутствующие в модели. Это повод для расследования, а не доказанная ошибка цены. |

Мы нашли разные ситуации: недостаток близких примеров, необычное сочетание признаков и цену, плохо согласующуюся с имеющимся описанием. Сам список больших остатков такого различия не показывал.

### От наблюдения — к следующему эксперименту

| Что наблюдаем | Какие действия имеют смысл |
|---|---|
| Дуга или другая форма в остатках | Проверить дополнительные признаки вида площади в квадрате или логарифма площади. Сравнить с исходной моделью на той же validation. |
| Разброс ошибок растёт вместе с ценой | Сравнить абсолютные и относительные ошибки по сегментам. Как отдельный эксперимент попробовать моделировать логарифм цены; после обратного преобразования всё равно оценить ошибки в долларах. |
| Систематический сдвиг в отдельной группе | Проверить признаки, которыми эта группа отличается, и их совместное действие: например, одинаково ли влияет площадь в разных районах. |
| У объекта мало похожих примеров | Проверить представленность такого типа объектов в train, расширить данные; не обещать ту же точность, что для частых объектов. |
| Цена резко отличается от цен близких по признакам домов | Проверить исходную запись, пропущенные характеристики и условия продажи. Удалять строку можно по установленной проблеме данных, а не только ради улучшения метрики. |

В таблице уже есть неиспользуемый нами признак `OverallQual` — общая оценка качества дома. Его добавление можно проверить как гипотезу; мы пока не установили, что именно он объясняет эти ошибки. [Описание исходных данных Ames](https://jse.amstat.org/v19n3/decock.pdf).

Проверяем изменения по одному и записываем результат. Если диагностика приводит к изменению модели, validation участвует в разработке. Test остаётся закрытым до завершения выбора; затем проверим устойчивость выводов с помощью кросс-валидации.

In [ ]:
data.columns

In [ ]:
data.OverallQual.describe()

In [ ]:
sns.scatterplot(data=data, x='OverallQual', y='SalePrice', alpha=0.3, color='black', edgecolor='black')

In [ ]:
data[data.Id == 1170].OverallQual

## 4. Как признаки входят в прогноз?

Для числовых признаков линейная модель имеет вид:

$$\widehat y=b+\sum_{j=1}^{p}w_j x_j,\qquad \Delta\widehat y=w_j\Delta x_j.$$

**Где:** $\widehat y$ — прогноз; $b$ — свободный член; $p$ — число признаков; $x_j$ — значение признака $j$; $w_j$ — его коэффициент. Во втором равенстве $\Delta x_j$ — изменение одного признака, $\Delta\widehat y$ — изменение прогноза **при неизменных остальных признаках**.

Коэффициент при площади в исходных единицах читается как доллары за дополнительный м² в прогнозе. Это зависимость внутри модели, не доказательство того, сколько принесёт реальная пристройка.

Наш `StandardScaler` передаёт в Ridge стандартизованные значения. Чтобы вернуть числовой коэффициент в исходные единицы:

$$z_j=\frac{x_j-\mu_j}{s_j},\qquad w_j^{\mathrm{raw}}=\frac{w_j^{\mathrm{scaled}}}{s_j}.$$

**Где:** $x_j$ — исходное числовое значение (после заполнения пропусков); $z_j$ — стандартизованное; $\mu_j$ и $s_j$ — сохранённые `mean_` и `scale_` из train; $w_j^{\mathrm{scaled}}$ — коэффициент Ridge при $z_j$; $w_j^{\mathrm{raw}}$ — коэффициент при изменении $x_j$ на одну исходную единицу. При записи всей модели в исходных координатах нужно также пересчитать свободный член; ниже восстанавливаем только числовые веса.

### Вспомним базовую категорию из One-Hot Encoding

В семинаре 2 «Подготовка данных» мы использовали `OneHotEncoder(drop="first")`. У каждого известного района есть бинарный индикатор, но **один из них удалён**. Иначе при наличии свободного члена возникла бы точная линейная зависимость: сумма всех индикаторов равна столбцу единиц.

**Базовая категория (reference category)** — район, чей индикатор удалён. Для него все оставшиеся индикаторы района равны нулю. Это точка отсчёта, а не «самый обычный» или «самый дешёвый» район. В нашем кодировщике категории упорядочены, и `drop="first"` удаляет первую из них; ниже напечатаем её название.

Коэффициент другого района — разница прогнозов с базовым районом при одинаковых числовых признаках. Например, коэффициент +10 000 означает более высокий на 10 000 долларов прогноз **этой модели** при замене базового района на данный.

Неизвестный кодировщику район при `handle_unknown="ignore"` тоже получает нулевые индикаторы. Это техническое правило; оно не доказывает, что его реальные цены соответствуют базовому району. [Документация OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [ ]:
def original_numeric_weights(fitted_model):
    fitted_scaler = fitted_model.named_steps["preprocess"].named_transformers_["num"].named_steps["scaler"]
    weights = fitted_model.named_steps["regression"].coef_[:len(numeric_features)]
    return pd.Series(weights / fitted_scaler.scale_, index=numeric_features)

feature_units = ["$/м²", "$/м²", "$/м²", "$/год", "$/м"]
coefficient_table = pd.DataFrame({
    "МНК: исходные единицы": original_numeric_weights(ols_model),
    "Ridge: исходные единицы": original_numeric_weights(ridge_model),
    "Единица коэффициента": feature_units,
})
display(coefficient_table.round(2))

fitted_preprocessor = model.named_steps["preprocess"]
fitted_encoder = fitted_preprocessor.named_transformers_["cat"]
reference_district = fitted_encoder.categories_[0][0]
district_weights = pd.Series(
    model.named_steps["regression"].coef_[len(numeric_features):],
    index=fitted_encoder.categories_[0][1:], name="Разница прогноза с базовым районом, $",
)
print("Базовая категория района:", reference_district)
display(district_weights.head(6).round(2).to_frame())

Коэффициент района описывает разницу прогноза относительно базовой категории при тех же числовых признаках. Показаны первые три категории по алфавиту; это не рейтинг важности.

> **Коэффициент, важность и статистическая значимость — разные характеристики.** Модуль коэффициента зависит от единиц измерения, набора других признаков и регуляризации. Даже после стандартизации он не показывает напрямую, насколько ухудшится качество без признака.

При сильной корреляции признаков изменение одного при неизменных остальных может описывать редкое или отсутствующее в данных сочетание. Например, увеличение жилой площади при неизменных характеристиках остальных помещений стоит сверить с реальными домами. [sklearn: интерпретация коэффициентов](https://scikit-learn.org/stable/auto_examples/inspection/plot_linear_model_coefficient_interpretation.html).

### Из чего сложился прогноз одного дома?

Оставим модель в её стандартизованных координатах и посчитаем каждое слагаемое: **коэффициент × подготовленное значение признака**.

Начало отсчёта — свободный член: прогноз для средних числовых значений train после импутации и базового района. Это условный дом; свободный член не обязан совпадать со средней ценой train. К нему добавим пять числовых вкладов и общий вклад района.

Разложение объясняет вычисление конкретного прогноза. Оно зависит от кодирования и точки отсчёта; это не разложение реальной рыночной стоимости на причины.

На графике синие слагаемые увеличивают прогноз относительно начала отсчёта, оранжевые — уменьшают.

In [ ]:
one_house = validation[features].iloc[[0]].copy()
prepared_house = fitted_preprocessor.transform(one_house)[0]
regression = model.named_steps["regression"]
all_contributions = prepared_house * regression.coef_
contributions = pd.Series(
    [*all_contributions[:len(numeric_features)], all_contributions[len(numeric_features):].sum()],
    index=features, name="Вклад в прогноз, $",
)
base_prediction = regression.intercept_
reconstructed_prediction = base_prediction + contributions.sum()
pipeline_prediction = model.predict(one_house)[0]
assert np.isclose(reconstructed_prediction, pipeline_prediction)

display(one_house)
display(pd.concat([
    pd.Series({"Свободный член": base_prediction}), contributions,
    pd.Series({"Итого: прогноз": reconstructed_prediction}),
]).round(2).to_frame("Сумма, $"))

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
ax.barh(contributions.index, contributions.values,
        color=np.where(contributions >= 0, "tab:blue", "tab:orange"))
ax.axvline(0, color="black", linewidth=1)
ax.invert_yaxis()
ax.set(xlabel="Слагаемое прогноза, $", ylabel="Признак",
       title=f"Начало: {base_prediction:,.0f} долл. → прогноз: {pipeline_prediction:,.0f} долл.")
plt.show()

## 5. Насколько устойчивы коэффициенты?

Веса оцениваются по обучающей выборке. Если заменить часть наблюдений, изменятся и данные, по которым решается задача МНК. Поэтому коэффициенты тоже могут измениться.

В семинаре 3 «Устойчивость линейной модели и Ridge» мы разбирали **мультиколлинеарность** — точную или приближённую линейную зависимость признаков. При сильной положительной корреляции двух почти одинаковых признаков увеличение одного веса можно компенсировать уменьшением другого. Для нескольких признаков мультиколлинеарность возможна и без очень высокой корреляции каждой пары.

Сначала воспроизведём этот эффект на синтетике, где он хорошо виден. Затем посмотрим, насколько он выражен в наших домах Ames. **Бутстрап** уже знаком: выбираем строки с возвращением и заново обучаем модель на полученной выборке.

### Сначала — два почти одинаковых признака

Создадим 80 объектов с двумя очень близкими признаками. Истинная зависимость известна:

$$y=2x_1+2x_2+\varepsilon.$$

**Где:** $x_1,x_2$ — признаки в одинаковых условных единицах; $y$ — целевая переменная в условных единицах; $\varepsilon$ — независимая случайная ошибка со средним 0 и стандартным отклонением 0,6. Истинные коэффициенты равны 2 и 2.

Повторим идею прошлого семинара. Если при неизменном свободном члене заменить веса на $w_1+t$ и $w_2-t$, изменение прогноза равно:

$$\Delta\widehat y=t(x_1-x_2).$$

**Где:** $\widehat y$ — прогноз модели; $\Delta\widehat y$ — его изменение; $t$ — прибавка к первому весу и противоположная прибавка ко второму. Когда $x_1$ и $x_2$ близки, веса могут заметно измениться и почти компенсировать друг друга.

120 раз выберем строки с возвращением. На каждой перевыборке заново обучим `StandardScaler → МНК` и `StandardScaler → Ridge`. Обе модели получают одни и те же строки; `alpha=5` фиксирован. Сравниваем веса в исходных единицах, а прогнозы — для одного и того же объекта $x_1=x_2=1$. Истинное среднее цели для него равно 4; конкретное наблюдение дополнительно содержало бы случайную ошибку.

В таблице SD (standard deviation) обозначает стандартное отклонение результатов по перевыборкам.

In [ ]:
#@title Устойчивость на двух почти одинаковых признаках { display-mode: "form" }
# Synthetic data: all quantities are in arbitrary units.
rng_stability = np.random.default_rng(42)
n_stability = 80
rho_stability = 0.999
x1_stability = rng_stability.normal(size=n_stability)
x2_stability = (
    rho_stability * x1_stability
    + np.sqrt(1 - rho_stability ** 2) * rng_stability.normal(size=n_stability)
)
X_stability = pd.DataFrame({"x1": x1_stability, "x2": x2_stability})
y_stability = 2 * x1_stability + 2 * x2_stability + rng_stability.normal(0, 0.6, n_stability)
object_stability = pd.DataFrame({"x1": [1.0], "x2": [1.0]})
true_mean_stability = 4.0

stability_templates = {
    "МНК": Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=5.0))]),
}

# Both methods see exactly the same bootstrap samples.
rng_resamples = np.random.default_rng(23)
stability_rows = []
for repeat in range(120):
    positions = rng_resamples.integers(0, n_stability, size=n_stability)
    for label, template in stability_templates.items():
        fitted = clone(template).fit(X_stability.iloc[positions], y_stability[positions])
        raw_weights = fitted.named_steps["model"].coef_ / fitted.named_steps["scale"].scale_
        stability_rows.append({
            "Повтор": repeat, "Модель": label,
            "w1": raw_weights[0], "w2": raw_weights[1],
            "Прогноз": fitted.predict(object_stability)[0],
        })
stability_demo = pd.DataFrame(stability_rows)
stability_demo["w1 + w2"] = stability_demo["w1"] + stability_demo["w2"]

fig, axs = plt.subplots(1, 3, figsize=(15, 4.8), layout="constrained")
observed_corr = X_stability.corr().iloc[0, 1]
axs[0].scatter(x1_stability, x2_stability, color="black", alpha=0.25, label="Обучающие объекты")
axs[0].scatter([1], [1], marker="*", s=180, color="tab:orange", edgecolor="black", label="Объект для прогноза")
axs[0].set(xlabel="x1, усл. ед. признака", ylabel="x2, усл. ед. признака",
           title=f"Два близких признака: r = {observed_corr:.4f}")
axs[0].set_aspect("equal", adjustable="box")
axs[0].legend(fontsize=8)

colors_stability = {"МНК": "0.4", "Ridge": "tab:blue"}
for label, group in stability_demo.groupby("Модель", sort=False):
    axs[1].scatter(group["w1"], group["w2"], s=20, alpha=0.5,
                   color=colors_stability[label], label=label)
weight_min = stability_demo[["w1", "w2"]].to_numpy().min() - 0.5
weight_max = stability_demo[["w1", "w2"]].to_numpy().max() + 0.5
weight_limits = [weight_min, weight_max]
axs[1].plot(weight_limits, 4 - np.array(weight_limits), "--", color="tab:orange",
            label="w1 + w2 = 4")
axs[1].scatter([2], [2], marker="x", s=80, color="black", linewidths=2, label="Истинные веса (2, 2)")
axs[1].set(xlim=weight_limits, ylim=weight_limits,
           xlabel="w1, усл. ед. цели / ед. признака", ylabel="w2, усл. ед. цели / ед. признака",
           title="120 переобучений: веса компенсируются")
axs[1].set_aspect("equal", adjustable="box")
axs[1].legend(fontsize=8)

rng_jitter = np.random.default_rng(5)
for row, label in enumerate(["МНК", "Ridge"]):
    values = stability_demo.loc[stability_demo["Модель"] == label, "Прогноз"].to_numpy()
    axs[2].scatter(values, row + rng_jitter.uniform(-0.12, 0.12, len(values)),
                   s=20, alpha=0.5, color=colors_stability[label])
axs[2].axvline(true_mean_stability, color="tab:orange", linestyle="--",
               label="Истинное среднее = 4")
axs[2].set_yticks([0, 1], ["МНК", "Ridge"])
axs[2].set(xlabel="Прогноз для x1 = x2 = 1, усл. ед. цели", ylabel="Модель",
           title="Тот же объект во всех повторах")
axs[2].legend(fontsize=8)
plt.show()

stability_summary = stability_demo.groupby("Модель", sort=False).agg(
    sd_w1=("w1", "std"), sd_w2=("w2", "std"), sd_sum=("w1 + w2", "std"),
    sd_prediction=("Прогноз", "std"), mean_prediction=("Прогноз", "mean"),
).rename(columns={
    "sd_w1": "SD веса w1", "sd_w2": "SD веса w2", "sd_sum": "SD суммы весов",
    "sd_prediction": "SD прогноза", "mean_prediction": "Средний прогноз",
})
display(stability_summary.round(3))

Слева видно, что признаки почти повторяют друг друга. В центре серая полоса показывает, как МНК перераспределяет веса между ними. Пунктир $w_1+w_2=4$ иллюстрирует их компенсацию; это не утверждение, что все оценки лежат точно на этой прямой. Ridge даёт гораздо более компактное облако.

Справа видно: при таком разбросе весов прогнозы для выбранного объекта остаются близкими друг к другу. В этом опыте средний прогноз Ridge смещён относительно известного значения 4. **Устойчивость коэффициентов не гарантирует более точного прогноза.**

Это та же геометрия, что в предыдущем семинаре: в центрированной задаче Ridge заменяет $X^\top X$ на $X^\top X+\alpha I$. Здесь $X$ — матрица подготовленных признаков, $\alpha$ — сила штрафа, $I$ — единичная матрица. Штраф ограничивает большие взаимно компенсирующиеся веса.

Это демонстрация чувствительности процедуры обучения, а не доверительные интервалы и не тест значимости.

### Теперь — реальные дома

Повторим 80 перевыборок train и каждый раз заново обучим весь Pipeline: импутер, скейлер, кодировщик и регрессию. Считаем отдельные продажи независимыми наблюдениями. МНК и Ridge получают одни и те же строки; `alpha=10` фиксирован.

Числовые коэффициенты каждый раз возвращаем в исходные единицы. Категории района могут выпадать из перевыборок, поэтому сравниваем пять числовых весов. **Первый рисунок показывает коэффициенты, отдельный второй рисунок — прогноз цены одного и того же дома `one_house`.**

In [ ]:
#@title Повторно обучаем модели на 80 bootstrap-выборках { display-mode: "form" }
rng_boot = np.random.default_rng(23)
B = 80
bootstrap_rows = []
for repeat in range(B):
    positions = rng_boot.integers(0, len(train), size=len(train))
    boot_train = train.iloc[positions]
    for name, template in [("МНК", ols_model), ("Ridge", ridge_model)]:
        fitted = clone(template).fit(boot_train[features], boot_train["SalePrice"])
        weights_raw = original_numeric_weights(fitted)
        bootstrap_rows.append({
            "Повтор": repeat, "Модель": name, **weights_raw.to_dict(),
            "Прогноз дома": fitted.predict(one_house)[0],
        })
bootstrap_results = pd.DataFrame(bootstrap_rows)

fig, axs = plt.subplots(1, 5, figsize=(17, 4), layout="constrained")
for ax, feature, unit in zip(axs, numeric_features, feature_units):
    sns.boxplot(data=bootstrap_results, x=feature, y="Модель", color="lightblue", ax=ax)
    ax.axvline(0, color="tab:orange", linestyle="--")
    ax.set(title=feature, xlabel=f"Коэффициент, {unit}", ylabel="Модель")
fig.suptitle("Ames: разброс пяти коэффициентов по 80 перевыборкам")
plt.show()

fig, ax = plt.subplots(figsize=(9, 3.5), layout="constrained")
sns.boxplot(data=bootstrap_results, x="Прогноз дома", y="Модель", color="lightblue", ax=ax)
ax.axvline(validation.loc[one_house.index[0], "SalePrice"], color="tab:orange", linestyle="--",
           label="Фактическая цена этого дома")
ax.set(xlabel="Прогноз цены, $", ylabel="Модель", title=f"Ames: один и тот же дом Id {int(validation.loc[one_house.index[0], 'Id'])}")
ax.legend(fontsize=9)
plt.show()

bootstrap_sd = bootstrap_results.groupby("Модель")[numeric_features + ["Прогноз дома"]].std().T
bootstrap_sd["Единица"] = feature_units + ["$"]
display(bootstrap_sd.round(2).rename_axis("SD по 80 перевыборкам"))

### Как прочитать этот результат

На Ames контраст слабее, чем на специально построенной синтетике. Это нормальный результат: мы не обязаны увидеть драматическое улучшение от Ridge на любых данных.

Для `YearBuilt` разброс коэффициента Ridge заметно меньше. Для других признаков уменьшение не столь выражено; у `LotFrontage` оценки могут менять знак. Взгляните и на таблицу SD: регуляризация не обязана уменьшать разброс **каждого** веса. При изменении метода обучения смещаются и сами оценки коэффициентов.

На отдельном рисунке обе модели чаще дают для этого дома прогнозы около 150 тыс. долларов, хотя его фактическая цена — 140 тыс. Перевыборки показывают разброс прогнозов, но не устраняют завышение цены этого объекта. Сравнивать ширину коробок разных коэффициентов между собой нельзя без учёта их единиц; сравниваем МНК и Ridge **для одного и того же признака**.

**Что делать дальше:** если знаки и величины весов нестабильны, проверить корреляционную матрицу и возможную линейную зависимость столбцов, сравнить модели без одного из дублирующих признаков, проверить несколько `alpha`. При этом контролировать и MAE/RMSE на validation: более устойчивые веса не гарантируют более точных прогнозов.

Разброс Ridge здесь не объявляется доверительным интервалом неизвестного «истинного веса» или тестом значимости. Повторный подбор `alpha` в этом опыте не учитывается.

## 6. Какие признаки использует готовая модель?

> **Перестановочная важность признака (permutation feature importance)** — изменение качества фиксированной обученной модели при случайном перемешивании значений одного признака между объектами оцениваемой выборки.

Перестановка сохраняет сами значения столбца, но нарушает их соответствие конкретным объектам. Модель, скейлер, импутер и кодировщик **не переобучаются**. Целевая переменная и остальные столбцы остаются на местах.

Если модель использовала площадь для прогноза, чужие площади должны мешать ей. Если столбец почти не использовался, изменение ошибки может быть маленьким. Посмотрим на синтетике: цена зависит от площади, а второй признак создан независимо как случайное число.

In [ ]:
rng_perm_toy = np.random.default_rng(18)
toy_X = pd.DataFrame({
    "Площадь": rng_perm_toy.uniform(40, 180, 400),
    "Случайный признак": rng_perm_toy.normal(size=400),
})
toy_y = 30_000 + 1_500 * toy_X["Площадь"] + rng_perm_toy.normal(0, 5_000, 400)
toy_fit_X, toy_check_X = toy_X.iloc[:250].copy(), toy_X.iloc[250:].copy()
toy_fit_y, toy_check_y = toy_y.iloc[:250], toy_y.iloc[250:]
toy_model = LinearRegression().fit(toy_fit_X, toy_fit_y)

toy_cases = {"Исходные данные": toy_check_X.copy()}
for column in toy_check_X.columns:
    shuffled = toy_check_X.copy()
    shuffled[column] = rng_perm_toy.permutation(shuffled[column].to_numpy())
    toy_cases[f"Перемешан: {column}"] = shuffled

fig, axs = plt.subplots(1, 3, figsize=(14, 4.5), layout="constrained")
for ax, (name, frame) in zip(axs, toy_cases.items()):
    prediction = toy_model.predict(frame)
    mae = mean_absolute_error(toy_check_y, prediction)
    ax.scatter(toy_check_y, prediction, color="black", alpha=0.2)
    ax.plot([60_000, 310_000], [60_000, 310_000], "--", color="tab:orange", label="Точное совпадение")
    ax.set(xlabel="Фактическая цена, $", ylabel="Прогноз, $",
           title=f"{name}\nMAE = {mae:,.0f} $", xlim=(60_000, 310_000), ylim=(60_000, 310_000))
axs[0].legend(fontsize=9)
plt.show()

### Измеряем ухудшение на домах Ames

Для MAE удобно считать увеличение ошибки:

$$I_j=\frac{1}{B}\sum_{r=1}^{B}\left(\operatorname{MAE}^{\mathrm{perm}}_{j,r}-\operatorname{MAE}_{\mathrm{base}}\right).$$

**Где:** $I_j$ — перестановочная важность признака $j$ в долларах; $B$ — число перестановок; $r$ — номер перестановки; $\operatorname{MAE}_{\mathrm{base}}$ — ошибка готовой модели на исходной validation; $\operatorname{MAE}^{\mathrm{perm}}_{j,r}$ — ошибка той же модели после перемешивания столбца $j$.

Сначала сделаем одну перестановку площади вручную. Копируем таблицу, чтобы сохранить исходную validation.

In [ ]:
rng_manual = np.random.default_rng(4)
shuffled_validation = validation[features].copy()
shuffled_validation["GrLivArea"] = rng_manual.permutation(
    shuffled_validation["GrLivArea"].to_numpy()
)
base_mae = mean_absolute_error(y_val, pred_val)
shuffled_mae = mean_absolute_error(y_val, model.predict(shuffled_validation))
display(pd.Series({
    "Исходная MAE, $": base_mae,
    "MAE после перемешивания площади, $": shuffled_mae,
    "Увеличение MAE, $": shuffled_mae - base_mae,
}).round(2).to_frame("Значение"))

Теперь повторим эксперимент для каждого признака через [`permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html).

В sklearn функции `scoring` устроены так, что больше — лучше. Поэтому задаём `scoring="neg_mean_absolute_error"`: качество равно **минус MAE**. Разность «исходное качество − качество после перестановки» тогда равна **увеличению MAE** из нашей формулы.

Передаём весь Pipeline и исходные шесть столбцов. Так `Neighborhood` перемешивается целиком, до One-Hot Encoding. Перемешивание отдельных бинарных индикаторов могло бы создать невозможный набор районов.

Сделаем 30 перестановок каждого столбца. В таблице **SD (standard deviation)** — стандартное отклонение полученных увеличений MAE.

In [ ]:
importance = permutation_importance(
    model, validation[features], y_val,
    scoring="neg_mean_absolute_error", n_repeats=30, random_state=17,
)
importance_table = pd.DataFrame({
    "Признак": features,
    "Среднее увеличение MAE, $": importance.importances_mean,
    "SD перестановок, $": importance.importances_std,
}).set_index("Признак").sort_values("Среднее увеличение MAE, $")
display(importance_table.round(2))

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
ax.errorbar(importance_table["Среднее увеличение MAE, $"], importance_table.index,
            xerr=importance_table["SD перестановок, $"], fmt="o", capsize=4, color="tab:blue")
ax.axvline(0, color="black", linestyle="--")
ax.set(xlabel="Увеличение MAE после перемешивания, $", ylabel="Признак",
       title="Ridge, validation: среднее ± SD по 30 перестановкам")
plt.show()

**Положительная важность:** перемешивание ухудшило прогноз. **Около нуля:** заметного ухудшения в этом опыте нет. **Отрицательная:** после перестановки ошибка уменьшилась; это может отражать случайный разброс или неудачное использование признака.

Отрезки на графике показывают стандартное отклонение результатов перестановок, **не доверительный интервал и не p-value**. Мы исследуем конкретную модель, метрику и проверочную выборку. Это не универсальный рейтинг свойств домов и не перестановочный тест гипотез из статистики.

Сильно скоррелированные признаки могут частично заменять друг друга при прогнозировании. Кроме того, перемешивание может создать нереалистичные сочетания, например площадь небольшого дома с характеристиками большого. Поэтому отдельные важности не всегда позволяют разделить общую информацию скоррелированных признаков. [sklearn: ограничения permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html#misleading-values-on-strongly-correlated-features).

**Что проверить:** если важность мала, сравнить модель после удаления признака и переобучения; если признаки сильно скоррелированы — проверить их совместное удаление или совместную перестановку строк этой группы. Если важность отрицательна, проверить повторяемость результата, качество значений и способ подготовки признака. Один результат на validation — основание для эксперимента, а не для автоматического удаления столбца.

### Что будет, если обучить модель без района?

**Удаление признака с переобучением (drop-column importance, ablation)** — другой эксперимент. Теперь остальные коэффициенты могут измениться и частично компенсировать удалённую информацию.

Для числовых признаков сохраняем ту же подготовку и `alpha=10`. Сравниваем на той же validation. Это сравнение двух заданных конфигураций; отдельно искать лучший `alpha` для каждой сейчас не будем.

In [ ]:
without_district = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regression", Ridge(alpha=10)),
])
without_district.fit(train[numeric_features], y_train)
without_district_mae = mean_absolute_error(
    y_val, without_district.predict(validation[numeric_features])
)
display(pd.Series({
    "Перемешать район, веса прежние": importance_table.loc["Neighborhood", "Среднее увеличение MAE, $"],
    "Удалить район, обучить заново": without_district_mae - base_mae,
}, name="Увеличение MAE, $").round(2).to_frame())

Числа не обязаны совпадать: в первом опыте готовая модель получает чужие районы; во втором учится обходиться без района. Маленькое ухудшение после удаления признака не доказывает, что его информация вообще не помогает предсказывать цель: его информацию могли заменить другие признаки.

## 7. Самостоятельная работа

### Задача 1. Ошибки по районам

В таблице `errors` уже есть район, остаток и абсолютная ошибка каждого дома validation.

1. Создайте таблицу `district_report`: для каждого района посчитайте число домов, MAE и средний остаток.
2. Оставьте районы, где не меньше 10 домов. Это учебное ограничение для читаемого сравнения, а не гарантия надёжности оценки.
3. Найдите среди них район с наибольшей MAE. По среднему остатку определите направление ошибки.
4. Объясните, почему средний остаток около нуля не гарантирует хорошего качества и почему район из двух домов нельзя уверенно объявить самым трудным для модели.

In [ ]:
district_report = None
# TODO: Group errors by Neighborhood and calculate count, MAE and mean residual.
# TODO: Keep groups with at least 10 houses and sort by MAE.
# Expected columns: "Число домов", "MAE, $", "Средний остаток, $".

if district_report is not None:
    display(district_report.round(1))

**Ваш вывод:** …

### Задача 2. Важность признаков у МНК

1. Вызовите `permutation_importance` для уже обученной `ols_model` на той же validation. Используйте `neg_mean_absolute_error`, 30 перестановок и `random_state=17`. Сохраните результат как `ols_importance`.
2. Выполните готовый код сравнения ниже. Для каких признаков важность у МНК и Ridge различается заметнее?
3. Выберите один признак и сформулируйте смысл его важности в долларах. Объясните, почему это число не является p-value.

Модель в этом опыте не переобучайте. Test не используйте.

In [ ]:
ols_importance = None
# TODO: Calculate permutation importance for the fitted OLS Pipeline on validation.

In [ ]:
#@title Сравнение после заполнения предыдущей ячейки { display-mode: "form" }
if ols_importance is None:
    print("Сначала вычислите ols_importance в задании 2.")
else:
    comparison = pd.DataFrame({
        "МНК": ols_importance.importances_mean,
        "Ridge": importance.importances_mean,
    }, index=features)
    display(comparison.round(2))
    fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
    positions = np.arange(len(features))
    for label, result, shift in [("МНК", ols_importance, -0.12), ("Ridge", importance, 0.12)]:
        ax.errorbar(result.importances_mean, positions + shift, xerr=result.importances_std,
                    fmt="o", capsize=3, label=label)
    ax.set_yticks(positions, features)
    ax.axvline(0, color="black", linestyle="--")
    ax.set(xlabel="Увеличение MAE после перемешивания, $", ylabel="Признак",
           title="Среднее ± SD по 30 перестановкам, validation")
    ax.legend()
    plt.show()

**Ваш вывод:** …

### Задача 3. Прочитайте коэффициент

Увеличьте жилую площадь дома `one_house` на 10 м², оставив остальные признаки прежними.

1. По коэффициенту Ridge в исходных единицах вычислите изменение прогноза.
2. Проверьте ответ разностью двух вызовов `predict`.
3. Объясните, почему совпадение вычислений ещё не доказывает, что реальная пристройка принесёт владельцу именно такую сумму.

In [ ]:
changed_house = one_house.copy()
manual_change = None
predicted_change = None
# TODO: Change GrLivArea by 10 square metres.
# TODO: Calculate the change using the original-unit Ridge coefficient.
# TODO: Compare predictions for the two feature vectors.

if manual_change is not None and predicted_change is not None:
    print(f"По коэффициенту: {manual_change:,.2f} $; по predict: {predicted_change:,.2f} $.")

### Итог: заключение о модели

Напишите короткий вывод, опираясь на полученные числа и графики:

* MAE и R² на validation: что они означают для этой задачи? Что показало сравнение с baseline?
* Одна обнаруженная проблема в ошибках.
* Один содержательно прочитанный коэффициент и наблюдение о его устойчивости.
* Один полезный для модели признак по permutation importance и ограничение такого вывода.
* Одно изменение данных или модели, которое вы предложили бы проверить дальше.

**Ваш вывод:** …

## Дополнительно. Статистическая значимость коэффициента

Предиктивная полезность — изменение качества прогноза. **Статистическая значимость (statistical significance)** — результат проверки гипотезы о неизвестном параметре при предположениях статистической модели.

Для обычного МНК можно проверять:

$$H_0:\beta_j=0,\qquad H_1:\beta_j\ne0.$$

**Где:** $H_0$ — нулевая гипотеза; $H_1$ — альтернативная; $\beta_j$ — неизвестный коэффициент признака $j$ в заданной статистической модели при фиксированном наборе остальных признаков. Его оценку по выборке обозначим $\widehat\beta_j$.

**Стандартная ошибка коэффициента (standard error, SE)** оценивает разброс его оценки при повторном получении данных. При нормальном приближении 95%-й доверительный интервал (confidence interval, CI) имеет вид:

$$\left[\widehat\beta_j-1{,}96\,\widehat{\mathrm{SE}}_j,\quad \widehat\beta_j+1{,}96\,\widehat{\mathrm{SE}}_j\right].$$

**Где:** $\widehat\beta_j$ — оценённый коэффициент; $\widehat{\mathrm{SE}}_j$ — его оценённая стандартная ошибка; 1,96 — квантиль стандартного нормального распределения для двустороннего интервала 95%. При выполнении условий метода около 95% таких интервалов в повторных исследованиях покрывали бы параметр; это не вероятность для параметра внутри уже рассчитанного интервала.

**P-value** — вероятность при верной нулевой гипотезе и предположениях теста получить значение тестовой статистики, не менее противоречащее нулевой гипотезе, чем наблюдаемое. Для двусторонней проверки рассматриваются отклонения в обе стороны. Результат называют статистически значимым, если p-value меньше заранее выбранного уровня значимости, например 0,05.

Ни маленький p-value, ни исключение нуля из интервала сами по себе не показывают большую предиктивную пользу или причинное влияние.

### Небольшой пример с известным устройством данных

Создадим два независимых признака. Цель равна `5 + 2*x1 + ошибка`: истинный коэффициент при `x1` равен 2, при `x2` — 0. Строки независимы; ошибки имеют условное среднее 0, а их разброс зависит от `x1`.

Используем `statsmodels` для обычного МНК. Настройка `HC3` оценивает стандартные ошибки с учётом возможного неравного разброса ошибок; `use_t=False` задаёт нормальное приближение. Интервалы и p-value здесь **приближённые**. HC3 не исправляет неверную форму зависимости, зависимость строк или выбор гипотез после просмотра результатов. [Документация statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLSResults.get_robustcov_results.html).

Этот отдельный пример относится к МНК. Его интервалы и p-value нельзя перенести на нашу Ridge-модель. Повторные перестановки в `permutation_importance` тоже не дают такой проверки.

In [ ]:
#@title Необязательный пример: коэффициент, SE, интервал и p-value { display-mode: "form" }
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("statsmodels") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels>=0.14,<0.16"])
import statsmodels.api as sm

rng_inference = np.random.default_rng(31)
X_inference = pd.DataFrame(rng_inference.normal(size=(400, 2)), columns=["x1", "x2"])
noise_scale = 0.4 + 0.8 * np.abs(X_inference["x1"])
y_inference = 5 + 2 * X_inference["x1"] + noise_scale * rng_inference.normal(size=400)
inference_design = sm.add_constant(X_inference)
ols_inference = sm.OLS(y_inference, inference_design).fit(cov_type="HC3", use_t=False)
intervals = ols_inference.conf_int(alpha=0.05)
inference_table = pd.DataFrame({
    "Коэффициент": ols_inference.params,
    "SE": ols_inference.bse,
    "Нижняя граница 95% CI": intervals[0],
    "Верхняя граница 95% CI": intervals[1],
    "p-value": ols_inference.pvalues,
})
display(inference_table)

fig, ax = plt.subplots(figsize=(9, 3.5), layout="constrained")
names = ["x1", "x2"]
centres = ols_inference.params.loc[names].to_numpy()
ax.errorbar(centres, [0, 1], xerr=1.96 * ols_inference.bse.loc[names].to_numpy(),
            fmt="o", capsize=5, label="Оценка и приближённый 95% CI")
ax.scatter([2, 0], [0, 1], marker="x", s=90, color="tab:orange", label="Параметр генератора")
ax.axvline(0, color="black", linestyle="--")
ax.set_yticks([0, 1], names)
ax.set(xlabel="Коэффициент, усл. ед. цели / усл. ед. признака", ylabel="Признак",
       title="Отдельная синтетика: обычный МНК, HC3")
ax.legend(fontsize=9)
plt.show()

## Что теперь должно получаться

**Метрики** описывают качество прогнозов; **остатки** помогают найти систематические проблемы; **коэффициенты** объясняют вычисление прогноза; **бутстрап** показывает чувствительность к обучающей выборке; **permutation importance** — зависимость качества готовой модели от конкретного признака.

Статистическая значимость проверяет гипотезу о параметре при дополнительных предположениях. Ни один из этих результатов сам по себе не устанавливает причинность.

Следующий шаг — проверить, насколько оценки качества и выбор модели зависят от разбиения данных: кросс-валидация.

## Источники

* [sklearn: R²](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html) — определение, отрицательные значения и постоянная цель.
* [NIST: диагностика модели по остаткам](https://www.itl.nist.gov/div898/handbook/pmd/section4/pmd44.htm).
* [sklearn: интерпретация коэффициентов линейных моделей](https://scikit-learn.org/stable/auto_examples/inspection/plot_linear_model_coefficient_interpretation.html) — единицы, мультиколлинеарность, устойчивость и ограничения причинных выводов.
* [sklearn: permutation feature importance](https://scikit-learn.org/stable/modules/permutation_importance.html) — смысл и ограничения метода.
* [sklearn: permutation_importance API](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html) — параметры и возвращаемые результаты.
* [sklearn: Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) — подготовка и модель как одна цепочка.
* [statsmodels: OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html) и [устойчивые оценки ковариации](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLSResults.get_robustcov_results.html) — дополнительный пример статистического вывода.
* [Ames / House Prices: описание данных](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data).
* [CMU: оптимальный прогноз и непредсказуемая ошибка](https://www.stat.cmu.edu/~siva/teaching/462/lec2.pdf).
* [Penn State: разложение сумм квадратов в МНК](https://online.stat.psu.edu/stat501/Lesson02).
* [Описание Ames автором датасета](https://jse.amstat.org/v19n3/decock.pdf).
* [Jupyter Widgets: интерактивные элементы управления](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html).